# Cycle 1 — Hyperparameter Tuning

The baseline modelling notebook trained all models with default settings. This notebook finds the optimal settings for each model using **Randomized Search Cross Validation**.

In [1]:
import sys, os                         
import pandas as pd                    
import numpy as np                     
import warnings
warnings.filterwarnings('ignore')      

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split

# RandomizedSearchCV: samples n_iter random param combos instead of exhaustive grid
# StratifiedKFold: cross-validation preserving class proportions in each fold

from sklearn.preprocessing import StandardScaler   # z-score scaler (used if needed for linear models)
from sklearn.metrics import accuracy_score, classification_report  # evaluation metrics
from xgboost import XGBClassifier                  # gradient boosting model to tune
from lightgbm import LGBMClassifier                # leaf-wise gradient boosting to tune


_here = os.getcwd()                                
while not os.path.isdir(os.path.join(_here, 'data')):  
    _p = os.path.dirname(_here)                        
    if _p == _here: raise RuntimeError('project root not found')  
    _here = _p
if _here not in sys.path:
    sys.path.insert(0, _here)                       

from config import Paths, ensure_dirs 
ensure_dirs()


df1 = pd.read_csv(str(Paths.PL_MATCHES_PROCESSED))  # load the preprocessed match dataset
X1 = df1.drop(columns=['FTR', 'Season'])              # features only — drop target and metadata
y1 = df1['FTR']                                       # target: 0=Away Win, 1=Draw, 2=Home Win

X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, random_state=42, stratify=y1  # stratify preserves class ratios in both splits
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # 5-fold CV for the inner search loop
print(f'Train: {len(X1_train)} | Test: {len(X1_test)} | Features: {X1_train.shape[1]}')


Train: 5472 | Test: 1368 | Features: 33


- Same splits as the modelling notebook (same `random_state=42`) — results are directly comparable
- `StratifiedKFold` ensures each fold has the same class distribution as the full dataset — (critical as we have data imbalance)

# Tuning XGBoost
Optimises XGBoost by testing 50 random combinations of key parameters using 5-fold stratified cross-validation. This efficiently improves performance over default settings while avoiding the high computational cost of an exhaustive GridSearch (250 vs 19,440 model fits).


In [2]:
xgb_param_grid = {
    'n_estimators':     [100, 200, 300, 500],    # more trees = more learning capacity (but slower)
    'max_depth':        [3, 4, 5, 6],             # tree depth: shallow=fast/underfit, deep=overfit
    'learning_rate':    [0.01, 0.05, 0.1, 0.2],  # step size per tree: low rate needs more trees
    'subsample':        [0.7, 0.8, 1.0],          # fraction of training rows sampled per tree
    'colsample_bytree': [0.7, 0.8, 1.0],          # fraction of features sampled per tree
    'min_child_weight': [1, 3, 5],                # min samples in a leaf — higher = more regularisation
    'gamma':            [0, 0.1, 0.2]             # min loss reduction needed for a split — higher = pruning
}

total_combinations = 4 * 4 * 4 * 3 * 3 * 3 * 3  # all possible param combinations
print(f'Total possible combinations: {total_combinations:,}')    # 3,888 — too many for grid search
print(f'Combinations we will try: 50 (RandomizedSearch)')        # we sample 50 at random
print(f'With 5-fold CV: 50 × 5 = 250 model fits')               # total number of model fits


Total possible combinations: 5,184
Combinations we will try: 50 (RandomizedSearch)
With 5-fold CV: 50 × 5 = 250 model fits


- Grid Search would try all 3,888 combinations × 5 folds = 19,440 fits — very slow
- RandomizedSearch tries 250 fits — much faster, finds comparably good results

#### Randomized Search

Runs 50 random combinations of hyperparameters, each evaluated with 5-fold CV. Returns the best combination.Resulting in finding a significantly better XGBoost configuration than the default settings.

In [3]:
xgb = XGBClassifier(random_state=42, eval_metric='mlogloss', verbosity=0)  # base model to tune

search_d1 = RandomizedSearchCV(
    xgb, xgb_param_grid,        # model and parameter space to search
    n_iter=50,                   # try 50 random combinations
    cv=cv,                       # 5-fold stratified cross-validation
    scoring='accuracy',          # optimise for accuracy (matches our evaluation metric)
    random_state=42,             # reproducible random sampling of param combinations
    n_jobs=-1,                   # parallelise across all available CPU cores
    verbose=1                    # print progress during fitting
)

search_d1.fit(X1_train, y1_train)  # run the search on the training set

print('Best hyperparameters:')
for param, value in search_d1.best_params_.items():
    print(f'  {param}: {value}')       # print each best parameter value
print()
print(f'Best CV accuracy: {search_d1.best_score_*100:.2f}%')  # cross-validation accuracy (on training folds)

y_pred_xgb_d1_tuned = search_d1.best_estimator_.predict(X1_test)  # predict with best found model
print(f'Test accuracy:    {accuracy_score(y1_test, y_pred_xgb_d1_tuned)*100:.2f}%')  # held-out test accuracy


Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best hyperparameters:
  subsample: 0.7
  n_estimators: 300
  min_child_weight: 1
  max_depth: 5
  learning_rate: 0.01
  gamma: 0
  colsample_bytree: 0.8

Best CV accuracy: 52.65%
Test accuracy:    53.65%


- **Low learning rate (0.01) + more trees (300)** — the tuner found that XGBoost learns better slowly on this dataset
- **subsample: 0.7** — using only 70% of data per tree reduces overfitting
- Test accuracy (53.65%) is above CV accuracy (52.65%) — the model generalises well

Improvement over baseline
- Untuned XGBoost: **50.95%**
- Tuned XGBoost: **53.65%%**
- **Gain: roughly +2 percentage points**

## Full Classification Report — Tuned XGBoost

In [4]:
print('TUNED XGBOOST')
print(f'Accuracy: {accuracy_score(y1_test, y_pred_xgb_d1_tuned)*100:.2f}%')  # overall test accuracy
print()
print(classification_report(y1_test, y_pred_xgb_d1_tuned, target_names=['Away Win', 'Draw', 'Home Win']))
# precision: of all predicted X, how many were correct
# recall: of all actual X, how many were found
# f1-score: harmonic mean of precision and recall


TUNED XGBOOST
Accuracy: 53.65%

              precision    recall  f1-score   support

    Away Win       0.55      0.45      0.49       383
        Draw       0.44      0.03      0.06       350
    Home Win       0.54      0.87      0.66       635

    accuracy                           0.54      1368
   macro avg       0.51      0.45      0.40      1368
weighted avg       0.51      0.54      0.46      1368



- Draw recall remains low (0.03) — even tuned XGBoost struggles with draws
- Home Win recall is high (0.87) — model confidently identifies home wins

## LightGBM Tuning

In [5]:
lgb_param_grid = {
    'n_estimators':     [100, 200, 300, 500],    # number of boosting rounds
    'learning_rate':    [0.01, 0.05, 0.1, 0.15], # step size — low values need more trees
    'max_depth':        [3, 4, 5, 6, 7],          # max tree depth (LightGBM also uses num_leaves)
    'num_leaves':       [20, 31, 50, 100],         # max leaves per tree — main complexity control in LightGBM
    'subsample':        [0.7, 0.8, 0.9, 1.0],     # row subsampling fraction per tree
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],     # feature subsampling fraction per tree
    'min_child_samples':[10, 20, 30],              # min samples per leaf — higher = less overfitting
    'reg_lambda':       [0, 0.1, 1.0]             # L2 regularisation on leaf weights
}

total_combinations_lgb = 4 * 4 * 5 * 4 * 4 * 4 * 3 * 3  # all possible combinations
print(f'Total possible LightGBM combinations: {total_combinations_lgb:,}')  # much larger space
print(f'Combinations we will try: 50 (RandomizedSearch)')
print(f'With 5-fold CV: 50 × 5 = 250 model fits')


Total possible LightGBM combinations: 46,080
Combinations we will try: 50 (RandomizedSearch)
With 5-fold CV: 50 × 5 = 250 model fits


In [6]:
lgb = LGBMClassifier(random_state=42, verbose=-1)  # base LightGBM model (verbose=-1 suppresses output)

search_lgb_d1 = RandomizedSearchCV(
    lgb, lgb_param_grid,         # model and parameter space
    n_iter=20,                    # 50 random combinations
    cv=cv,                        # same 5-fold stratified CV used for XGBoost
    scoring='accuracy',           # optimise for accuracy
    random_state=42,              # reproducible sampling
    n_jobs=-1,                    # parallelise across all CPU cores
    verbose=1                     # print progress
)

search_lgb_d1.fit(X1_train, y1_train)  # run the search

print('Best hyperparameters:')
for param, value in search_lgb_d1.best_params_.items():
    print(f'  {param}: {value}')        # print each winning parameter
print()
print(f'Best CV accuracy: {search_lgb_d1.best_score_*100:.2f}%')  # best cross-validation accuracy

y_pred_lgb_d1_tuned = search_lgb_d1.best_estimator_.predict(X1_test)  # predict with best model
print(f'Test accuracy:    {accuracy_score(y1_test, y_pred_lgb_d1_tuned)*100:.2f}%')  # held-out test accuracy


Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best hyperparameters:
  subsample: 0.7
  reg_lambda: 1.0
  num_leaves: 100
  n_estimators: 200
  min_child_samples: 20
  max_depth: 4
  learning_rate: 0.01
  colsample_bytree: 0.7

Best CV accuracy: 52.32%
Test accuracy:    53.29%


In [7]:
print('TUNED LIGHTGBM')
print(f'Accuracy: {accuracy_score(y1_test, y_pred_lgb_d1_tuned)*100:.2f}%')  # overall test accuracy
print()
print(classification_report(y1_test, y_pred_lgb_d1_tuned, target_names=['Away Win', 'Draw', 'Home Win']))
# precision/recall/f1 per class — draw remains the hardest class to predict


TUNED LIGHTGBM
Accuracy: 53.29%

              precision    recall  f1-score   support

    Away Win       0.54      0.43      0.48       383
        Draw       0.50      0.01      0.02       350
    Home Win       0.53      0.89      0.66       635

    accuracy                           0.53      1368
   macro avg       0.52      0.44      0.39      1368
weighted avg       0.53      0.53      0.45      1368

